In [65]:
from pyalex import Works, Authors, Sources, Institutions, Topics, Publishers, Funders
from pyalex import config
import pyalex
import os
import pandas as pd
from tqdm import tqdm
import json

In [66]:
pyalex.config.email = "cosmoskaluga@yandex.ru"

In [67]:
config.max_retries = 2
config.retry_backoff_factor = 0.1
config.retry_http_codes = [429, 500, 503]

## SIRT6-related papers

In [70]:
query_terms = ['"SIRT6"', 
               '"sirt6"', 
               '"Sirt6"', 
               '"Sirtuin 6"', 
               '"Sirtuin6"'
              ]

search_query = " OR ".join(query_terms)

In [71]:
search_query

'"SIRT6" OR "sirt6" OR "Sirt6" OR "Sirtuin 6" OR "Sirtuin6"'

In [72]:
year_filter = ">1950"

In [77]:
def get_paginated_entries(work_obj):
    all_papers = []

    works_generator = work_obj.paginate(per_page=200)

    for page_num, page in enumerate(works_generator, 1):
        print(f"Processing page {page_num} with {len(page)} papers...")
        for work in page:
            paper_info = {
                'mag': work.get('ids', 'N/A').get('mag', 'N/A'),
                'title': work.get('title', 'N/A'),
                'publication_year': work.get('publication_year', 'N/A'),
                'doi': work.get('doi', 'N/A'),
                'pmid': work.get('ids', 'N/A').get('pmid', 'N/A'),
                'authors': [author['author']['display_name'] for author in work.get('authorships', [])],
                'journal': work.get('primary_location', {}).get('raw_source_name', 'N/A'),
                'abstract': work['abstract'], #work.get('abstract', {}),
                'citation_count': work.get('cited_by_count', 0),
                'open_access': work.get('open_access', {}).get('is_oa', False),
                'open_access_link': work.get('open_access', {}).get('oa_url', False),
                'type': work.get('type', 'N/A'),
                'is_retracted': work.get('is_retracted', 'N/A'),
                'concepts': work.get('concepts', {}),
                'topics': work.get('topics', {}),
                'page': page_num
            }
            all_papers.append(paper_info)

    return pd.DataFrame(all_papers)

In [78]:
title_q = Works().filter(title={"search": search_query}, publication_year=year_filter).sort(cited_by_count="desc")
abstract_q = Works().filter(abstract={"search": search_query}, publication_year=year_filter).sort(cited_by_count="desc")

In [79]:
title_paginated = get_paginated_entries(title_q)
abstract_paginated = get_paginated_entries(abstract_q)

Processing page 1 with 200 papers...
Processing page 2 with 200 papers...
Processing page 3 with 200 papers...
Processing page 4 with 200 papers...
Processing page 5 with 200 papers...
Processing page 6 with 200 papers...
Processing page 7 with 200 papers...
Processing page 8 with 200 papers...
Processing page 9 with 98 papers...
Processing page 10 with 0 papers...
Processing page 1 with 200 papers...
Processing page 2 with 200 papers...
Processing page 3 with 200 papers...
Processing page 4 with 200 papers...
Processing page 5 with 200 papers...
Processing page 6 with 200 papers...
Processing page 7 with 200 papers...
Processing page 8 with 200 papers...
Processing page 9 with 200 papers...
Processing page 10 with 200 papers...
Processing page 11 with 200 papers...
Processing page 12 with 99 papers...
Processing page 13 with 0 papers...


In [80]:
filtered_papers = pd.concat([pd.DataFrame(title_paginated), pd.DataFrame(abstract_paginated)]) \
    .query("is_retracted == False") \
    .query("type.isin(['article', 'preprint'])") \
    .drop_duplicates(subset=['title'])

In [81]:
filtered_papers.shape

(2025, 16)

In [93]:
no_abstracts_df = filtered_papers.loc[filtered_papers.abstract.isnull()]
no_abstracts_df.loc[no_abstracts_df.pmid == "N/A"]

,mag,title,publication_year,doi,pmid,authors,journal,abstract,citation_count,open_access,open_access_link,type,is_retracted,concepts,topics,page
13,2064802388,SIRT6 represses LINE1 retrotransposons by ribo...,2014,https://doi.org/10.1038/ncomms6011,N/A,"[Michael Van Meter, Mehr Kashyap, Sarallah Rez...",Nature Communications,None,420,True,https://www.nature.com/articles/ncomms6011.pdf,article,False,"[{'id': 'https://openalex.org/C7029365', 'wiki...","[{'id': 'https://openalex.org/T10434', 'displa...",1
33,2033340092,Liver cancer initiation is controlled by AP-1 ...,2012,https://doi.org/10.1038/ncb2590,N/A,"[Lihua Min, Yuan Ji, Latifa Bakiri, Zhixin Qiu...",Nature Cell Biology,None,247,False,None,article,False,"[{'id': 'https://openalex.org/C2775975398', 'w...","[{'id': 'https://openalex.org/T11051', 'displa...",1
191,1891886702,Sirtuin 6 regulates glucose-stimulated insulin...,2015,https://doi.org/10.1007/s00125-015-3778-2,N/A,"[Xiwen Xiong, Gaihong Wang, Rongya Tao, Pengfe...",Diabetologia,None,72,True,https://link.springer.com/content/pdf/10.1007/...,article,False,"[{'id': 'https://openalex.org/C2777595374', 'w...","[{'id': 'https://openalex.org/T11051', 'displa...",1
223,2174684589,Enhanced insulin sensitivity in skeletal muscl...,2015,https://doi.org/10.1016/j.molmet.2015.09.003,N/A,"[Jason Anderson, Giorgio Ramadori, Rafael M. I...",Molecular Metabolism,None,61,True,https://doi.org/10.1016/j.molmet.2015.09.003,article,False,"[{'id': 'https://openalex.org/C126322002', 'wi...","[{'id': 'https://openalex.org/T11051', 'displa...",2
299,N/A,The effects of caloric restriction and a high-...,2012,https://doi.org/10.1007/bf03654792,N/A,"[Lili Luo, Xiaochun Chen, Yu‐Cai Fu, Jin-Jie X...",Aging Clinical and Experimental Research,None,47,False,None,article,False,"[{'id': 'https://openalex.org/C134018914', 'wi...","[{'id': 'https://openalex.org/T11051', 'displa...",2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2019,N/A,"Effects of Follicle-Stimulating Hormone, Insul...",2023,https://doi.org/10.2139/ssrn.4412915,N/A,"[Evandro Carlos Archilia, Camilo Andrés Peña B...",None,None,0,True,http://dx.doi.org/10.2139/ssrn.4412915,preprint,False,"[{'id': 'https://openalex.org/C147708747', 'wi...","[{'id': 'https://openalex.org/T12964', 'displa...",11
2029,N/A,A DPOC como uma doença de envelhecimento acele...,2009,None,N/A,[Fátima Rodrigues],"Revista Portuguesa de Pneumologia, Vol 15, Iss...",None,0,True,https://doaj.org/article/1efbd6fcc87548bfa163b...,article,False,"[{'id': 'https://openalex.org/C71924100', 'wik...","[{'id': 'https://openalex.org/T10143', 'displa...",11
2044,N/A,Enhancer-Driven Shh Signaling Promotes Glia-to...,2023,https://doi.org/10.2139/ssrn.4630699,N/A,"[Xin Shen, Hang Zhang, Zesheng Song, Yangjiele...",None,None,0,True,http://dx.doi.org/10.2139/ssrn.4630699,preprint,False,"[{'id': 'https://openalex.org/C2776340970', 'w...","[{'id': 'https://openalex.org/T10269', 'displa...",11
2102,N/A,Enhancement of Porcine Embryo Developmental Co...,2024,https://doi.org/10.2139/ssrn.4827347,N/A,"[Jin-Gu No, Seokho Kim, Haesun Lee, Tae‐Uk Kwa...",None,None,0,True,http://dx.doi.org/10.2139/ssrn.4827347,preprint,False,"[{'id': 'https://openalex.org/C2776188440', 'w...","[{'id': 'https://openalex.org/T10773', 'displa...",11


In [94]:
filtered_papers.to_csv("sirt6_paper_corpus/SIRT6_openalex_papers_all_entries.csv", index=False)

In [53]:
filtered_papers.abstract.isnull().value_counts()

abstract
False    1312
True      713
Name: count, dtype: int64

In [55]:
available_abstracts_df = filtered_papers.loc[~filtered_papers.abstract.isnull()]

In [60]:
available_abstracts_df[['title', 'publication_year', 'doi', 'abstract']].to_csv("sirt6_paper_corpus/SIRT6_openalex_available_abstracts.csv", index=False)

In [120]:
papers_dict = {}

for idx, row in filtered_papers.iterrows():
    paper_id = row.get("title", f'paper_{idx}')
    papers_dict[paper_id] = row["concepts"]

with open("papers_with_concepts_articles_preprints.json", "w", encoding="utf-8") as f:
    json.dump(papers_dict, f, indent = 4, ensure_ascii = False)